In [1]:
%pip install librosa
%pip install scipy
%pip install ffmpeg
%pip install mediapipe
%pip install decord --upgrade
%pip install spikingjelly


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import scipy
import scipy.signal
import torch
import librosa
import matplotlib.pyplot as plt
import torch.nn as nn
from torch import Tensor
from collections.abc import Sequence
from functools import partial
from spikingjelly.activation_based import neuron, surrogate
from spikingjelly.activation_based import functional
from typing import Any, Callable, Optional, Union
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp
from tqdm.notebook import tqdm
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
from decord import VideoReader, cpu


# Step 1: Visual - Frontend Processing

## 1. Chuyển từ 30fps sang 25fps


In [8]:
!ffmpeg -i "Dataset_Output\_Địa chấn_ lao động toàn cầu： Người trẻ buông chuột để làm lao động chân tay _ VTV24\00016\video.mp4" -r 25 -s 1920x1080 -vcodec libx264 -pix_fmt yuv420p -crf 23 -preset fast "Speech Reconstruction/tester.mp4" -y

ffmpeg version 8.1-essentials_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers
  built with gcc 15.2.0 (Rev11, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-cairo --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --enable-libass --enable-libfreetype --enable-libfribidi --enable-libharfbuzz --enable-libvidstab --enable-libvmaf --enable-libzimg --enable-amf --enable-cuda-llvm --enable-cuvid --enable-dxva2 --enable-d3d11va --enable-d3d12va --enable-ffnvcodec --enable-libvpl --enable-nvdec --enable-nvenc --enable-vaapi --enable-openal --enable-libgme --enable-libopenmpt --enable-libopenco

## 2. Chuyển từ 48khz sang 16khz


In [10]:
!ffmpeg -hide_banner -loglevel error -y -i "Speech Reconstruction\tester.mp4" -c:v copy -acodec aac -ar 16000 -ac 1 "Speech Reconstruction\tester16khz.mp4"


## 3. Trích xuất Mouth ROI bằng Frame Landmaker

In [3]:
!curl -o face_landmarker_v2_with_blendshapes.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100  3.58M 100  3.58M   0      0  6.27M      0                              0
100  3.58M 100  3.58M   0      0  6.27M      0                              0
100  3.58M 100  3.58M   0      0  6.27M      0                              0


### Face Mesh and npz

In [ ]:
def draw_landmarks_on_image(rgb_image, detection_result):
  face_landmarks_list = detection_result.face_landmarks
  annotated_image = np.copy(rgb_image)

  # Loop through the detected faces to visualize.
  for idx in range(len(face_landmarks_list)):
    face_landmarks = face_landmarks_list[idx]

    # Draw the face landmarks.


    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=face_landmarks,
        connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION,
        landmark_drawing_spec=None,
        connection_drawing_spec=drawing_styles.get_default_face_mesh_tesselation_style())
    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=face_landmarks,
        connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_CONTOURS,
        landmark_drawing_spec=None,
        connection_drawing_spec=drawing_styles.get_default_face_mesh_contours_style())
    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=face_landmarks,
        connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_LEFT_IRIS,
          landmark_drawing_spec=None,
          connection_drawing_spec=drawing_styles.get_default_face_mesh_iris_connections_style())
    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=face_landmarks,
        connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_RIGHT_IRIS,
          landmark_drawing_spec=None,
          connection_drawing_spec=drawing_styles.get_default_face_mesh_iris_connections_style())

  return annotated_image

def plot_face_blendshapes_bar_graph(face_blendshapes):
  # Extract the face blendshapes category names and scores.
  face_blendshapes_names = [face_blendshapes_category.category_name for face_blendshapes_category in face_blendshapes]
  face_blendshapes_scores = [face_blendshapes_category.score for face_blendshapes_category in face_blendshapes]
  # The blendshapes are ordered in decreasing score value.
  face_blendshapes_ranks = range(len(face_blendshapes_names))

  fig, ax = plt.subplots(figsize=(12, 12))
  bar = ax.barh(face_blendshapes_ranks, face_blendshapes_scores, label=[str(x) for x in face_blendshapes_ranks])
  ax.set_yticks(face_blendshapes_ranks, face_blendshapes_names)
  ax.invert_yaxis()

  # Label each bar with values
  for score, patch in zip(face_blendshapes_scores, bar.patches):
    plt.text(patch.get_x() + patch.get_width(), patch.get_y(), f"{score:.4f}", va="top")

  ax.set_xlabel('Score')
  ax.set_title("Face Blendshapes")
  plt.tight_layout()
  plt.show()

In [ ]:
# STEP 2: Cấu hình FaceLandmarker
base_options = python.BaseOptions(model_asset_path='face_landmarker_v2_with_blendshapes.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=True,
    running_mode=vision.RunningMode.VIDEO, # Chế độ tối ưu cho video
    num_faces=1)

# Khởi tạo VideoReader
video_path = "Speech Reconstruction/tester16khz.mp4"
vr = VideoReader(video_path, ctx=cpu(0))
fps = 25

# Setup VideoWriter để vẽ thử kết quả (Bước 2.3)
w, h = int(vr[0].shape[1]), int(vr[0].shape[0])
out = cv2.VideoWriter('Speech Reconstruction/tester16khz_landmakers.mp4',
                         cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

all_results = []

# Dùng context manager để tự động giải phóng detector
with vision.FaceLandmarker.create_from_options(options) as detector:
    for i in range(len(vr)):
        frame = vr[i].asnumpy() # Decord trả về RGB

        # Chuyển sang định dạng mp.Image
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)

        # Tính toán timestamp (ms) - 25fps ứng với 40ms/frame
        frame_timestamp_ms = int(i * (1000 / fps))

        # Dùng detect_for_video
        result = detector.detect_for_video(mp_image, frame_timestamp_ms)
        all_results.append(result)

        annotated_frame = draw_landmarks_on_image(frame, result)
        out.write(cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR))

out.release()
print(f"{len(all_results)} frames!")

### Save landmaker to npz

In [ ]:
import numpy as np

def save_landmarks_to_npz(all_results, output_path, source_name):
    all_landmarks = []
    all_blendshapes = []

    for result in all_results:
        if result and result.face_landmarks:
            # Lấy 468 điểm (x, y, z)
            current_frame = [[lm.x, lm.y, lm.z] for lm in result.face_landmarks[0]]
            all_landmarks.append(current_frame)

            # Lấy 52 chỉ số Blendshapes
            if result.face_blendshapes:
                scores = [b.score for b in result.face_blendshapes[0]]
                all_blendshapes.append(scores)
            else:
                all_blendshapes.append(np.zeros(52))
        else:
            # Frame lỗi hoặc không thấy mặt
            all_landmarks.append(np.zeros((468, 3)))
            all_blendshapes.append(np.zeros(52))

    # Chuyển thành mảng NumPy
    all_landmarks_np = np.array(all_landmarks)
    all_blendshapes_np = np.array(all_blendshapes)

    # Lưu file nén
    np.savez_compressed(
        output_path,
        landmarks=all_landmarks_np,
        blendshapes=all_blendshapes_np,
        metadata={'fps': 25, 'source': source_name}
    )
    print(f"{len(all_landmarks_np)} frames vào {output_path}")

In [ ]:
save_landmarks_to_npz(all_results, 'Speech Reconstruction/tester16khz_Full_Data.npz', 'Speech Reconstruction/tester16khz.mp4')

374 frames vào Speech Reconstruction/tester16khz_Full_Data.npz


### Affine transform


In [3]:
def affine_trans(landmarks, frame, target_size=(224, 224)):
  h, w, _ = frame.shape
  left = np.array([landmarks[33][0] * w, landmarks[33][1] * h])
  right = np.array([landmarks[263][0] * w, landmarks[263][1] * h])

  dY = right[1] - left[1]
  dX = right[0] - left[0]

  angle = np.degrees(np.arctan2(dY, dX))

  theta = ((left[0] + right[0]) / 2, (left[1] + right[1]) / 2)

  distance = np.sqrt((left[0] - right[0])**2 + (left[1] - right[1])**2)
  ratio = target_size[0] * 0.3
  scale = ratio / distance

  Matrix = cv2.getRotationMatrix2D(theta, angle, scale)
  Matrix[0, 2] += (target_size[0] * 0.5) - theta[0]
  Matrix[1, 2] += (target_size[1] * 0.35) - theta[1]

  transform = cv2.warpAffine(frame, Matrix, target_size, flags=cv2.INTER_CUBIC)

  return transform

### Đọc file npz landmaker và áp dụng Affine

In [4]:
def read_timestamp(video_path, landmark_path):
  data = np.load(landmark_path, allow_pickle=True)
  landmarks = data['landmarks']
  vr = VideoReader(video_path, ctx=cpu(0))
  out = cv2.VideoWriter('Speech Reconstruction/tester16khz_aff.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 25, (224, 224))
  for i in range(len(vr)):
    frame = vr[i].asnumpy()
    landmark = landmarks[i]

    if landmark is not None:
      transform = affine_trans(landmark, frame)
      out.write(cv2.cvtColor(transform, cv2.COLOR_RGB2BGR))
    else:
      out.write(np.zeros((h, w, 3), dtype=np.uint8))
  out.release()


In [ ]:
read_timestamp('Speech Reconstruction/tester16khz.mp4', 'Speech Reconstruction/tester16khz_Full_Data.npz')

### Trích xuất môi


In [4]:
def lips(transform, landmark_path, size=112):
      data = np.load(landmark_path, allow_pickle=True)
      landmarks = data['landmarks']
      midX = (landmarks[61][0] + landmarks[291][0]) / 2
      midY = (landmarks[0][1] + landmarks[17][1]) / 2

      half = size // 2
      
      y1, y2 = max(0, int(midY - half)), int(midY + half)
      x1, x2 = max(0, int(midX - half)), int(midX + half)
      
      lip_roi = transform[y1:y2, x1:x2]

      # Nếu cắt bị thiếu do chạm biên, resize lại cho đủ 112x112
      if lip_roi.shape[0] != size or lip_roi.shape[1] != size:
            lip_roi = cv2.resize(lip_roi, (size, size))
      
      # if len(lip_roi.shape) == 3:
      #       lip_roi = cv2.cvtColor(lip_roi, cv2.COLOR_BGR2GRAY)

      return lip_roi

## 4. ĐỒNG BỘ STFT -> Mel-Spectrogram

In [4]:
def get_aligned_mel(audio_path,  num_video_frames, sr=16000, n_mels=80):
    y, _ = librosa.load(audio_path, sr=sr)

    hop_length = int(sr / fps)
    n_fft = 1024

    mel_spec = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels
    )

    log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
    T_audio = log_mel_spec.shape[1]

    if T_audio > num_video_frames:
        log_mel_spec = log_mel_spec[:, :num_video_frames]
    elif T_audio < num_video_frames:
        padding = num_video_frames - T_audio
        last_frame = log_mel_spec[:, -1:]
        pad_matrix = np.repeat(last_frame, padding, axis=1)
        log_mel_spec = np.concatenate([log_mel_spec, pad_matrix], axis=1)
    
    assert log_mel_spec.shape[1] == num_video_frames, "Số khung hình âm thanh không khớp với số khung hình video sau khi căn chỉnh."
    return log_mel_spec



# def get_aligned_mel(audio_path, fps, num_video_frames, sr=16000, n_mels=80):
#     y, _ = librosa.load(audio_path, sr=sr)  # vẫn load bằng librosa
#     y_tensor = torch.from_numpy(y).float().unsqueeze(0)  # (1, T)

#     mel_spec, _ = mel_spectogram(
#         audio=y_tensor,
#         sample_rate=sr,
#         hop_length=int(sr / fps),
#         win_length=1024,
#         n_fft=1024,
#         n_mels=n_mels,
#         f_min=0.0,
#         f_max=8000.0,
#         power=1,
#         normalized=False,
#         min_max_energy_norm=True,
#         norm="slaney",
#         mel_scale="slaney",
#         compression=True
#     )
#     mel_spec = mel_spec.squeeze(0).cpu().numpy()  # (n_mels, T)
#     # cắt/pad để đồng bộ với video
#     T_audio = mel_spec.shape[1]
#     if T_audio > num_video_frames:
#         mel_spec = mel_spec[:, :num_video_frames]
#     elif T_audio < num_video_frames:
#         padding = num_video_frames - T_audio
#         pad_matrix = np.repeat(mel_spec[:, -1:], padding, axis=1)
#         mel_spec = np.concatenate([mel_spec, pad_matrix], axis=1)
#     return mel_spec


def get_aligned_mel(audio_path, fps, num_video_frames, sr=16000, n_mels=80):
    """Chuyển audio -> mel spectrogram bằng librosa, config khớp speechbrain."""
    y, _ = librosa.load(audio_path, sr=sr)
    
    hop_length = 256
    
    # STFT -> Mel (power=1 => dùng magnitude, không phải power)
    mel_spec = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=1024,
        hop_length=hop_length,
        win_length=1024,
        n_mels=n_mels,
        fmin=0.0,
        fmax=8000.0,
        power=1,              # magnitude spectrogram (khớp speechbrain power=1)
        norm="slaney",         # khớp norm="slaney"
        htk=False,             # htk=False = mel_scale="slaney"
    )
    
    # compression=True trong speechbrain = log1p
    mel_spec = np.log1p(mel_spec)
    
    # min_max_energy_norm=True trong speechbrain = normalize về [0, 1]
    mel_min = mel_spec.min()
    mel_max = mel_spec.max()
    if mel_max - mel_min > 1e-8:
        mel_spec = (mel_spec - mel_min) / (mel_max - mel_min)
    
    # Cắt/pad để đồng bộ với video
    T_audio = mel_spec.shape[1]
    if T_audio > num_video_frames:
        mel_spec = mel_spec[:, :num_video_frames]
    elif T_audio < num_video_frames:
        padding = num_video_frames - T_audio
        pad_matrix = np.repeat(mel_spec[:, -1:], padding, axis=1)
        mel_spec = np.concatenate([mel_spec, pad_matrix], axis=1)
    
    return mel_spec  # shape: (n_mels, T)


# STEP 2. SPIKING VISUAL ENCODER

## Direct Encoding

In [5]:

class SpikingDirectEncoder(nn.Sequential):
    def __init__(self, in_channels=1, out_channels=64) -> None:
        super().__init__()
        self.spatial_conv = nn.Conv3d(in_channels, 45, kernel_size=(1, 7, 7), stride=(1, 2, 2), padding=(0, 3, 3), bias=False)
        self.bn1 = nn.BatchNorm3d(45)
        self.lif1 = neuron.LIFNode(tau=2.0, surrogate_function=surrogate.ATan())

        self.temporal_conv = nn.Conv3d(45, out_channels, kernel_size=(5, 1, 1), stride=(1, 1, 1), padding=(2, 0, 0), bias=False)
        self.bn2 = nn.BatchNorm3d(out_channels)
        self.lif2 = neuron.LIFNode(tau=2.0, surrogate_function=surrogate.ATan())

    def forward(self, X):
        # X: (B, C, T, H, W)
        s1 = self.spatial_conv(X)
        b1 = self.bn1(s1)
        l1 = self.lif1(b1)

        s2 = self.temporal_conv(l1)
        b2 = self.bn2(s2)
        l2 = self.lif2(b2)
        return l2

## ResNet 2D Plus 1D feature extraction

In [6]:
class SpikingConv2DPlus1D(nn.Sequential):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: int, stride: int=1, padding: int=1):
        super().__init__(
          nn.Conv3d(
              in_channels=in_channels,
              out_channels=mid_channels,
              kernel_size=(1, 3, 3),
              stride=(1, stride, stride),
              padding=(0, padding, padding),
              bias=False,
          ),
          nn.BatchNorm3d(mid_channels),
          neuron.LIFNode(surrogate_function=surrogate.ATan()),
          nn.Conv3d(
              in_channels=mid_channels,
              out_channels=out_channels,
              kernel_size=(3, 1, 1),
              stride=(1, 1, 1),
              padding=(1, 0, 0),
              bias=False,
          ),
        )
    @staticmethod
    def get_stride(stride: int):
        return stride, stride, stride



In [7]:
class SpikingBasicBlock(nn.Module):
    expansion: int = 1
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        conv_builder: Callable[..., nn.Module],
        stride: int = 1,
        downsample: Optional[nn.Module] = None,
    ) -> None:
        mid_channels = (in_channels * out_channels * 3 * 3 * 3) // (in_channels * out_channels * 1 * 3 * 3 + in_channels * out_channels * 3 * 1 * 1)
        super().__init__()
        self.conv1 = nn.Sequential(
            conv_builder(in_channels, out_channels, mid_channels, stride),
            nn.BatchNorm3d(out_channels),
            neuron.LIFNode(tau=2.0, surrogate_function=surrogate.ATan())
        )
        self.conv2 = nn.Sequential(
            conv_builder(out_channels, out_channels, mid_channels),
            nn.BatchNorm3d(out_channels),
        )
        self.lif = neuron.LIFNode(tau=2.0, surrogate_function=surrogate.ATan())
        self.downsample = downsample
        self.stride = stride

    def forward(self, X: Tensor) -> Tensor:
        residual = X
        out = self.conv1(X)
        out = self.conv2(out)
        if self.downsample is not None:
            residual = self.downsample(X)
        out += residual
        out = self.lif(out)
        return out


In [8]:
from torch.utils.checkpoint import checkpoint
from spikingjelly.activation_based import functional

class VidResNet(nn.Module):
    def __init__(
        self,
        block = SpikingBasicBlock,
        conv_makers = [SpikingConv2DPlus1D] * 4,
        layers = [2, 2, 2, 2],
        zero_init_residual: bool = False,
    ):
        super().__init__()
        self.in_channels = 64
        self.stem = SpikingDirectEncoder(in_channels=1, out_channels=64)

        self.layer1 = self._make_layer(block, 64, conv_makers[0], layers[0], stride=1)
        self.layer2 = self._make_layer(block, 128, conv_makers[1], layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, conv_makers[2], layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, conv_makers[3], layers[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool3d((None, 1, 1))
        self.fc = nn.Linear(512 * block.expansion, 512)

        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, SpikingBasicBlock):
                    nn.init.constant_(m.bn3.weight, 0)

    # ★ Helper: checkpoint an toàn cho SNN
    def _snn_checkpoint(self, layer, X):
        """Gradient checkpointing có reset neuron states."""
        def _forward(x):
            functional.reset_net(layer)  # Reset v, spike của LIF/PLIF trong layer
            return layer(x)
        return checkpoint(_forward, X, use_reentrant=True)

    def forward(self, X: Tensor) -> Tensor:
        X = self.stem(X)
        # ★ Checkpoint từng block, không phải từng layer
        for block in self.layer1:
            X = self._snn_checkpoint(block, X)
        for block in self.layer2:
            X = self._snn_checkpoint(block, X)
        for block in self.layer3:
            X = self._snn_checkpoint(block, X)
        for block in self.layer4:
            X = self._snn_checkpoint(block, X)
        X = self.avgpool(X)
        X = X.squeeze(-1).squeeze(-1)
        X = X.permute(0, 2, 1)
        X = self.fc(X)
        return X

    def _make_layer(
        self,
        block: SpikingBasicBlock,
        out_channels: int,
        conv_builder: Callable[..., nn.Module],
        num_blocks: int,
        stride: int,
    ) -> nn.Sequential:
        downsample = None

        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv3d(self.in_channels, out_channels * block.expansion, kernel_size=1, stride=(1, stride, stride), bias=False),
                nn.BatchNorm3d(out_channels * block.expansion)
            )
        layers = []
        layers.append(block(self.in_channels, out_channels, conv_builder, stride, downsample))

        self.in_channels = out_channels * block.expansion
        for i in range(1, num_blocks):
            layers.append(block(self.in_channels, out_channels, conv_builder))

        return nn.Sequential(*layers)


## Spiking Transformer (S - ViT)

## Attention

In [9]:
class SpikingAttention(nn.Module):
    def __init__(self, z_dim=512, n_heads=8):
        super().__init__()
        self.n_heads = n_heads
        self.dim_head = z_dim // n_heads
        self.scale = self.dim_head ** -0.5

        self.q_linear = nn.Linear(z_dim, z_dim)
        self.k_linear = nn.Linear(z_dim, z_dim)
        self.v_linear = nn.Linear(z_dim, z_dim)

        self.q_plif = neuron.ParametricLIFNode(init_tau=2.0, surrogate_function=surrogate.ATan())
        self.k_plif = neuron.ParametricLIFNode(init_tau=2.0, surrogate_function=surrogate.ATan())
        self.v_plif = neuron.ParametricLIFNode(init_tau=2.0, surrogate_function=surrogate.ATan())

        self.attn_plif = neuron.ParametricLIFNode(init_tau=2.0, surrogate_function=surrogate.ATan())
        self.proj = nn.Linear(z_dim, z_dim)

    def forward(self, X):
        B, T, C = X.shape

        # Q, K, V
        q = self.q_linear(X)                # (B, T, C)
        k = self.k_linear(X)
        v = self.v_linear(X)

        # Đưa qua PLIF (flatten theo thời gian)
        q = q.reshape(B * T, C)
        q = self.q_plif(q)
        q = q.reshape(B, T, C)

        k = k.reshape(B * T, C)
        k = self.k_plif(k)
        k = k.reshape(B, T, C)

        v = v.reshape(B * T, C)
        v = self.v_plif(v)
        v = v.reshape(B, T, C)

        # Multi-head reshape
        q = q.reshape(B, T, self.n_heads, self.dim_head).permute(0, 2, 1, 3)  # (B, n_heads, T, dim_head)
        k = k.reshape(B, T, self.n_heads, self.dim_head).permute(0, 2, 1, 3)
        v = v.reshape(B, T, self.n_heads, self.dim_head).permute(0, 2, 1, 3)

        # Attention scores
        attn_scores = (q @ k.transpose(-2, -1)) * self.scale   # (B, n_heads, T, T)
        attn_out = attn_scores @ v                             # (B, n_heads, T, dim_head)

        # Merge heads
        attn_out = attn_out.permute(0, 2, 1, 3).reshape(B, T, C)  # (B, T, C)

        # attn_plif
        attn_out = attn_out.reshape(B * T, C)
        attn_out = self.attn_plif(attn_out)
        attn_out = attn_out.reshape(B, T, C)

        # Projection
        out = self.proj(attn_out)          # (B, T, C)
        return out

## MLP

In [10]:
class SpikingMLP(nn.Module):
    def __init__(self, in_dim=512, hidden_dim=2048, out_dim=512):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)

        self.plif = neuron.ParametricLIFNode(init_tau=2.0, surrogate_function=surrogate.ATan())
        
        self.fc2 = nn.Linear(hidden_dim, out_dim)
        self.bn2 = nn.BatchNorm1d(out_dim)

    def forward(self, X):
        # X: (B, T, C)
        B, T, C = X.shape

        # fc1
        X = self.fc1(X)                     # (B, T, hidden_dim)
        # Chuẩn bị cho BN1 (yêu cầu (N, C, L) với L=1)
        X = X.reshape(B * T, -1)            # (B*T, hidden_dim)
        X = X.unsqueeze(-1)                 # (B*T, hidden_dim, 1)
        X = self.bn1(X)                     # BN trên kênh hidden_dim
        X = X.squeeze(-1)                   # (B*T, hidden_dim)

        # Neuron PLIF (xử lý từng vector độc lập)
        X = self.plif(X)                    # (B*T, hidden_dim)
        X = X.reshape(B, T, -1)             # (B, T, hidden_dim)

        # fc2
        X = self.fc2(X)                     # (B, T, out_dim)
        # BN2
        X = X.reshape(B * T, -1)            # (B*T, out_dim)
        X = X.unsqueeze(-1)                 # (B*T, out_dim, 1)
        X = self.bn2(X)
        X = X.squeeze(-1)                   # (B*T, out_dim)
        X = X.reshape(B, T, -1)             # (B, T, out_dim)

        return X

## SpikeTransformer

In [11]:
class SpikingVisionTransformer(nn.Module):
    def __init__(self, z_dim=512, n_heads=8, mlp_hidden_dim=2048):
        super().__init__()
        self.attn = SpikingAttention(z_dim=z_dim, n_heads=n_heads)
        self.attn_plif = neuron.ParametricLIFNode(init_tau=2.0, surrogate_function=surrogate.ATan())

        self.mlp = SpikingMLP(in_dim=z_dim, hidden_dim=mlp_hidden_dim, out_dim=z_dim)
        self.mlp_plif = neuron.ParametricLIFNode(init_tau=2.0, surrogate_function=surrogate.ATan())

    def forward(self, X):
        # X: (B, T, C)
        # Attention
        attn_out = self.attn(X)                     # (B, T, C)
        residual = attn_out + X                      # residual connection

        # attn_plif
        residual_flat = residual.reshape(-1, residual.size(-1))  # (B*T, C)
        residual_spike = self.attn_plif(residual_flat)
        residual_spike = residual_spike.reshape(residual.shape)  # (B, T, C)

        # MLP
        mlp_out = self.mlp(residual_spike)           # (B, T, C)
        residual2 = mlp_out + residual_spike          # residual connection

        # mlp_plif
        residual2_flat = residual2.reshape(-1, residual2.size(-1))  # (B*T, C)
        out_spikes = self.mlp_plif(residual2_flat)
        out_spikes = out_spikes.reshape(residual2.shape)            # (B, T, C)

        return out_spikes

## ALIF NODE

In [12]:
class ConfALIFNode(nn.Module):
    def __init__(self, tau_m=2.0, tau_a=1.5, beta=0.1):
        super().__init__()
        self.tau_m = tau_m
        self.tau_a = tau_a
        self.beta = beta
        self.surrogate = surrogate.ATan()
        self.threshold_base = 1.0

    def forward(self, X):
        B, T, C = X.shape
        v_mem = torch.zeros(B, C, device=X.device)
        a_adapt = torch.zeros(B, C, device=X.device)
        spikes_seq = []
        v_mem_seq = []
        for t in range(T):
            current_I = X[:, t, :]
            a_adapt = a_adapt * (1 - 1/self.tau_a)
            v_th = self.threshold_base + a_adapt
            v_mem = v_mem * (1 - 1/self.tau_m) + current_I
            v_mem_seq.append(v_mem)
            spike = self.surrogate(v_mem - v_th)
            v_mem = v_mem * (1 - spike)
            a_adapt = a_adapt + self.beta * spike
            spikes_seq.append(spike)
        return torch.stack(spikes_seq, dim=1), torch.stack(v_mem_seq, dim=1)


# STEP 3: Readout Layer

In [13]:
class ReadoutLayer(nn.Module):
    def __init__(self, in_dim=512, out_dim=512, tau_a=1.5, tau_m=2.0, beta=0.1):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)
        self.alif = ConfALIFNode(tau_m=tau_m, tau_a=tau_a, beta=beta)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, X): # X: B T C 
        proj_X = self.proj(X)
        _, v_mem_seq = self.alif(proj_X)
        z_continuous = self.norm(v_mem_seq)
        return z_continuous      

## SpikeTransformerEncoder

In [14]:
class SpikingViTEncoder(nn.Module):
    def __init__(self, z_dim=512, n_heads=8, n_blocks=8, mlp_hidden_dim=2048, T_max=1000):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1, T_max, z_dim))
        self.backbone = VidResNet()
        self.transformers = nn.ModuleList([
            SpikingVisionTransformer(z_dim=z_dim, n_heads=n_heads, mlp_hidden_dim=mlp_hidden_dim) 
            for _ in range(n_blocks)
        ])
        self.readout = ReadoutLayer(in_dim=z_dim, out_dim=z_dim)
    
    def forward(self, X): # X: B C T H W
        features = self.backbone(X) # B T C
        features += self.pos_embedding[:, :features.size(1), :]
        for block in self.transformers:
            features = block(features) # B T C
        z_continuous = self.readout(features) # B T C
        return z_continuous

# STEP 4: CONTINUOUS INR DECODER

## FiLM Layer

In [15]:
class FiLMLayer(nn.Module):
    def __init__(self, z_dim=512, condition_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(condition_dim, condition_dim),
            nn.ReLU(),
            nn.Linear(condition_dim, 2 * z_dim)
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.constant_(self.net[-1].bias[:z_dim], 0.0)
        nn.init.constant_(self.net[-1].bias[z_dim:], 1.0)
    
    def forward(self, X, Condition):
        gamma, beta = self.net(Condition).chunk(2, dim=-1)

        shape = [1] * (X.ndim - 2) 
        gamma = gamma.view(gamma.size(0), -1, *shape)
        beta = beta.view(beta.size(0), -1, *shape)
        return X * gamma + beta

## From FiLM Layer to TFiLM Layer

In [16]:
class TFiLM(nn.Module):
    def __init__(self, z_dim=512, condition_dim=512):
        super().__init__()
        self.film = FiLMLayer(z_dim=z_dim, condition_dim=condition_dim)

    def forward(self, X, Condition):
        B, T = X.shape[:2]
        X_flat = X.reshape(B * T, *X.shape[2:])
        Condition_flat = Condition.reshape(B * T, *Condition.shape[2:])
        out_flat = self.film(X_flat, Condition_flat)
        out = out_flat.reshape(B, T, *X.shape[2:])
        return out

## SIREN Network

In [17]:
class FiLMSIREN(nn.Module):
    def __init__(self, in_features, out_features, omega_zero=30.0, is_first=False):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.omega_zero = omega_zero
        self.is_first = is_first
        with torch.no_grad():
            if self.is_first:
                bound = 1 / in_features
            else:
                bound = torch.sqrt(torch.tensor(6.0 / in_features)) / self.omega_zero
            
            self.linear.weight.uniform_(-bound, bound)

    def forward(self, X, gamma, beta):
        X = self.linear(X)
        if self.is_first:
            X = self.omega_zero * X
        out = gamma * X + beta
        return torch.sin(out)

## TFiLM + SIREN -> DECODER

In [ ]:
class TFiLMSIRENDecoder(nn.Module):
    def __init__(self, condition_dim=512, hidden_dim=256, out_dim=80, num_layers=4, omega_zero=30.0, use_conv=False):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        total_params = num_layers * 2 * hidden_dim
        if use_conv:
            self.param_net = nn.Conv1d(condition_dim, total_params, kernel_size=3, padding=1)
            nn.init.zeros_(self.param_net.weight)
            with torch.no_grad():
                for i in range(num_layers):
                    start = i * 2 * hidden_dim
                    self.param_net.bias[start:start + hidden_dim].fill_(0.0)
                    self.param_net.bias[start + hidden_dim:start + 2 * hidden_dim].fill_(1.0)
        else:
            self.param_net = nn.Sequential(
                nn.Linear(condition_dim, condition_dim),
                nn.ReLU(),
                nn.Linear(condition_dim, total_params)
            )
            nn.init.zeros_(self.param_net[-1].weight)
            with torch.no_grad():
                for i in range(num_layers):
                    start = i * 2  * hidden_dim
                    self.param_net[-1].bias[start:start + hidden_dim].fill_(0.0)
                    self.param_net[-1].bias[start + hidden_dim:start + 2 * hidden_dim].fill_(1.0)

        self.siren_layers = nn.ModuleList()
        self.siren_layers.append(FiLMSIREN(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, is_first=True))
        for _ in range(1, num_layers):
            self.siren_layers.append(FiLMSIREN(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, is_first=False))
        self.final_layer = nn.Linear(hidden_dim , out_dim)
        nn.init.xavier_uniform_(self.final_layer.weight)
        nn.init.zeros_(self.final_layer.bias)

        self.input_constant = nn.Parameter(torch.randn(1, hidden_dim) * 0.1)
    
    def forward(self, Condition):
        B, T, _ = Condition.shape
        if isinstance(self.param_net, nn.Conv1d):
            params = self.param_net(Condition.permute(0, 2, 1))
            params = params.permute(0, 2, 1)
        else:
            params = self.param_net(Condition.reshape(B * T, -1))
            params = params.reshape(B, T, -1)
        
        gammas = []
        betas = []

        for i in range(self.num_layers):
            start = i * 2 * self.hidden_dim
            beta = params[:, :, start: start + self.hidden_dim]
            gamma = params[:, :, start + self.hidden_dim: start + 2 * self.hidden_dim]
            gammas.append(gamma)
            betas.append(beta)
        
        X = self.input_constant.expand(B * T, -1) # B*T, hidden_dim
        gammas_flat = [g.reshape(B * T, -1) for g in gammas]
        betas_flat = [b.reshape(B * T, -1) for b in betas]

        for i, layer in enumerate(self.siren_layers):
            X = layer(X, gammas_flat[i], betas_flat[i])
        out = self.final_layer(X)
        out = out.reshape(B, T, -1)
        return out

In [19]:
# test_model.py
import torch
# Import các class của bạn

# Tạo dummy input
dummy_video = torch.randn(2, 1, 10, 112, 112)   # (B, C, T, H, W)
encoder = SpikingViTEncoder(
    z_dim=512,
    n_heads=8,
    n_blocks=2,               # thử nghiệm nhẹ hơn
    mlp_hidden_dim=2048,
    T_max=1000
)
decoder = TFiLMSIRENDecoder(
    condition_dim=512,        # phải khớp với z_dim của encoder
    hidden_dim=256,
    out_dim=80,
    num_layers=4,
    omega_zero=30.0,
    use_conv=False
)

z = encoder(dummy_video)          # (B, T, cond_dim)
mel_pred = decoder(z)            # (B, T, out_dim)
print("Encoder output:", z.shape)
print("Decoder output:", mel_pred.shape)

Encoder output: torch.Size([2, 10, 512])
Decoder output: torch.Size([2, 10, 80])


# Save .pt for Dataset

In [22]:
import os
import subprocess
from tqdm.notebook import tqdm
import glob

DATASET_PATH = "Dataset_Output"  # Thư mục gốc
OUTPUT_PATH = "Processed_Data"   # Nơi lưu tensor
FPS = 25
SR = 16000
N_MELS = 80
HOP_LENGTH = int(SR / FPS)  

def convert_video_fps(input_path, output_path, target_fps=25):
    subprocess.run([
        "ffmpeg", "-i", input_path,
        "-r", str(target_fps),
        "-vcodec", "libx264",
        "-pix_fmt", "yuv420p",
        "-crf", "23",
        "-preset", "fast",
        output_path, "-y"
    ], check=True, capture_output=True)

def affine_trans(landmarks, frame, target_size=(224, 224)):
    """Căn chỉnh gương mặt."""
    h, w, _ = frame.shape
    left = np.array([landmarks[33][0] * w, landmarks[33][1] * h])
    right = np.array([landmarks[263][0] * w, landmarks[263][1] * h])

    dY = right[1] - left[1]
    dX = right[0] - left[0]
    angle = np.degrees(np.arctan2(dY, dX))
    theta = ((left[0] + right[0]) / 2, (left[1] + right[1]) / 2)

    distance = np.sqrt((left[0] - right[0])**2 + (left[1] - right[1])**2)
    ratio = target_size[0] * 0.3
    scale = ratio / distance

    Matrix = cv2.getRotationMatrix2D(theta, angle, scale)
    Matrix[0, 2] += (target_size[0] * 0.5) - theta[0]
    Matrix[1, 2] += (target_size[1] * 0.35) - theta[1]

    transform = cv2.warpAffine(frame, Matrix, target_size, flags=cv2.INTER_CUBIC)
    return transform

def get_lips(transform, landmarks, size=112):
    """Cắt vùng môi, tương tự notebook."""
    midX = int((landmarks[61][0] + landmarks[291][0]) / 2 * transform.shape[1])
    midY = int((landmarks[0][1] + landmarks[17][1]) / 2 * transform.shape[0])
    half = size // 2

    y1, y2 = max(0, midY - half), midY + half
    x1, x2 = max(0, midX - half), midX + half

    lip_roi = transform[y1:y2, x1:x2]
    if lip_roi.shape[0] != size or lip_roi.shape[1] != size:
        lip_roi = cv2.resize(lip_roi, (size, size))
    return lip_roi

def get_aligned_mel(audio_path, fps, num_video_frames, sr=16000, n_mels=80):
    """Chuyển audio -> mel spectrogram bằng librosa, config khớp speechbrain."""
    y, _ = librosa.load(audio_path, sr=sr)
    
    hop_length = 256
    
    # STFT -> Mel (power=1 => dùng magnitude, không phải power)
    mel_spec = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=1024,
        hop_length=hop_length,
        win_length=1024,
        n_mels=n_mels,
        fmin=0.0,
        fmax=8000.0,
        power=1,              # magnitude spectrogram (khớp speechbrain power=1)
        norm="slaney",         # khớp norm="slaney"
        htk=False,             # htk=False = mel_scale="slaney"
    )
    
    # compression=True trong speechbrain = log1p
    mel_spec = np.log1p(mel_spec)
    
    # min_max_energy_norm=True trong speechbrain = normalize về [0, 1]
    mel_min = mel_spec.min()
    mel_max = mel_spec.max()
    if mel_max - mel_min > 1e-8:
        mel_spec = (mel_spec - mel_min) / (mel_max - mel_min)
    
    # Cắt/pad để đồng bộ với video
    T_audio = mel_spec.shape[1]
    if T_audio > num_video_frames:
        mel_spec = mel_spec[:, :num_video_frames]
    elif T_audio < num_video_frames:
        padding = num_video_frames - T_audio
        pad_matrix = np.repeat(mel_spec[:, -1:], padding, axis=1)
        mel_spec = np.concatenate([mel_spec, pad_matrix], axis=1)
    
    return mel_spec  # shape: (n_mels, T)



base_options = python.BaseOptions(model_asset_path='face_landmarker_v2_with_blendshapes.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    running_mode=vision.RunningMode.VIDEO,
    num_faces=1
)

# ==========================================
# 3. Vòng lặp xử lý chính
# ==========================================


# Tìm tất cả file video.mp4 trong toàn bộ dataset
video_files = glob.glob(os.path.join(DATASET_PATH, "**", "video.mp4"), recursive=True)

folders = [os.path.dirname(v) for v in video_files]  # thư mục chứa video

print(f"Tìm thấy {len(folders)} folder có video.mp4")


for folder_path in tqdm(folders, "Đang xử lí"):
    print(f"Xử lý: {folder_path}")
    video_files = glob.glob(os.path.join(folder_path, "video.*"))
    audio_files = glob.glob(os.path.join(folder_path, "audio.*"))
    if not video_files:
        print(f"  -> Không tìm thấy file video trong {folder_path}")
        continue
    if not audio_files:
        print(f"  -> Không tìm thấy file audio trong {folder_path}")
        continue
    video_path = video_files[0]   
    audio_path = audio_files[0] 

    temp_video_path = os.path.join(folder_path, "temp_25fps.mp4")
    convert_video_fps(video_path, temp_video_path, target_fps=25)
    video_path = temp_video_path   # ghi đè để dùng file đã chuyển fps
    
    detector = vision.FaceLandmarker.create_from_options(options)

    try:
        # ----------------------------------------------------
        # Đọc video và trích xuất môi
        # ----------------------------------------------------
        vr = VideoReader(video_path, ctx=cpu(0))
        num_frames = len(vr)
        lip_frames = []

        for i in range(num_frames):
            frame = vr[i].asnumpy()
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
            timestamp_ms = int(i * (1000 / FPS))
            result = detector.detect_for_video(mp_image, timestamp_ms)

            if result.face_landmarks:
                landmarks = result.face_landmarks[0]
                lm_list = [[lm.x, lm.y, lm.z] for lm in landmarks]
                aligned = affine_trans(lm_list, frame)
                lip = get_lips(aligned, lm_list)
                lip_frames.append(lip)
            else:
                # Nếu không tìm thấy mặt, thêm frame trắng (giữ nguyên kích thước)
                if len(lip_frames) > 0:
                    lip_frames.append(np.zeros_like(lip_frames[-1]))
                else:
                    lip_frames.append(np.zeros((112, 112, 3), dtype=np.uint8))

        if len(lip_frames) == 0:
            raise RuntimeError("Không trích xuất được khung môi nào")
        del vr
        import gc
        gc.collect()

        # Chuyển đổi sang tensor
        lip_video = np.array([cv2.cvtColor(l, cv2.COLOR_RGB2GRAY) for l in lip_frames])
        lip_tensor = torch.from_numpy(lip_video).float() / 255.0   # (T, 112, 112)
        lip_tensor = lip_tensor.unsqueeze(0)                      # (1, T, 112, 112)

        # ----------------------------------------------------
        # Xử lý audio -> mel
        # ----------------------------------------------------
        mel_np = get_aligned_mel(audio_path, FPS, lip_tensor.shape[1])
        mel_tensor = torch.from_numpy(mel_np).float()              # (80, T)
        mel_tensor = mel_tensor.permute(1, 0)                     # (T, 80)

        # ----------------------------------------------------
        # Lưu file
        # ----------------------------------------------------
        path_parts = folder_path.split(os.sep)
        if len(path_parts) >= 2:
            safe_name = f"{path_parts[-2]}_{path_parts[-1]}"
        else:
            safe_name = path_parts[-1]
            
        save_dict = {
            'video': lip_tensor,
            'mel': mel_tensor
        }
        
        
        # Nếu thư mục Processed_Data chưa có thì tạo mới
        os.makedirs(OUTPUT_PATH, exist_ok=True)
        
        out_path = os.path.join(OUTPUT_PATH, f"{safe_name}.pt")
        torch.save(save_dict, out_path)
        print(f"  -> Đã lưu {safe_name}.pt | Shape: {lip_tensor.shape}, {mel_tensor.shape}")
        os.remove(temp_video_path)

    except Exception as e:
        print(f"  -> LỖI: {e}")
        # Có thể thêm traceback để chi tiết hơn:
        import traceback
        traceback.print_exc()
        continue

Tìm thấy 2439 folder có video.mp4


Đang xử lí:   0%|          | 0/2439 [00:00<?, ?it/s]

Xử lý: Dataset_Output\Mưa đá, giông lốc gây thiệt hại tại nhiều địa phương _ VTV24\00002
  -> Đã lưu Mưa đá, giông lốc gây thiệt hại tại nhiều địa phương _ VTV24_00002.pt | Shape: torch.Size([1, 29, 112, 112]), torch.Size([29, 80])
Xử lý: Dataset_Output\Mưa đá, giông lốc gây thiệt hại tại nhiều địa phương _ VTV24\00005
  -> Đã lưu Mưa đá, giông lốc gây thiệt hại tại nhiều địa phương _ VTV24_00005.pt | Shape: torch.Size([1, 77, 112, 112]), torch.Size([77, 80])
Xử lý: Dataset_Output\Mưa đá, giông lốc gây thiệt hại tại nhiều địa phương _ VTV24\00012
  -> Đã lưu Mưa đá, giông lốc gây thiệt hại tại nhiều địa phương _ VTV24_00012.pt | Shape: torch.Size([1, 228, 112, 112]), torch.Size([228, 80])
Xử lý: Dataset_Output\Mất 2,3 tỷ đồng sau khi được người quen qua mạng rủ đánh bạc trực tuyến _ VTV24\00016
  -> Đã lưu Mất 2,3 tỷ đồng sau khi được người quen qua mạng rủ đánh bạc trực tuyến _ VTV24_00016.pt | Shape: torch.Size([1, 169, 112, 112]), torch.Size([169, 80])
Xử lý: Dataset_Output\Mất 750 

# DataLoader

In [20]:
from torch.utils.data import Dataset, DataLoader
class VNLipDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = data_dir 
        self.files = [f for f in os.listdir(self.data_dir) if f.endswith('.pt')]
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        file_path = os.path.join(self.data_dir, self.files[idx])
        data = torch.load(file_path, weights_only=False)

        video = data['video']
        mel = data['mel']

        return video, mel

## Collate function (pad T)

In [21]:
def collate_pad(batch):
    videos, mels = zip(*batch) #video: B T C H W #mel T 80
    lengths = torch.tensor([v.shape[1] for v in videos])
    T_max = lengths.max().item()

    padding_vids = []
    padding_mels = []

    for v, m in zip(videos, mels):
        T_v = v.shape[1]
        T_m = m.shape[0]

        if T_v < T_max:
            pad_v = torch.zeros(v.shape[0], T_max - T_v, v.shape[2], v.shape[3])
            v = torch.cat([v, pad_v], dim=1)
        if T_m < T_max:
            pad_m = torch.zeros(T_max - T_m, m.shape[1])
            m = torch.cat([m, pad_m], dim=0)
        
        padding_vids.append(v)
        padding_mels.append(m)

    video_batches = torch.stack(padding_vids, dim=0)
    mel_batches = torch.stack(padding_mels, dim=0)

    return video_batches, mel_batches, lengths

## DataLoader


In [22]:
import os
dataset = VNLipDataset('Processed_Data')

dataloader = DataLoader(
    dataset=dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    collate_fn=collate_pad,
)


## Main process

In [23]:
encoder = SpikingViTEncoder() 
decoder = TFiLMSIRENDecoder()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder.to(device)
decoder.to(device)

TFiLMSIRENDecoder(
  (param_net): Sequential(
    (0): Linear(in_features=512, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=2048, bias=True)
  )
  (siren_layers): ModuleList(
    (0-3): 4 x FiLMSIREN(
      (linear): Linear(in_features=256, out_features=256, bias=True)
    )
  )
  (final_layer): Linear(in_features=256, out_features=80, bias=True)
)

In [24]:
def train_one_epoch(
        encoder: nn.Module,
        decoder: nn.Module, 
        dataloader: DataLoader, 
        optimizer: torch.optim.Optimizer,
        criterion: nn.Module,
        device: torch.device,
        use_mask: bool=True,
        max_grad_norm: Optional[float]=1
    ):
    encoder.train()
    decoder.train()
    total_loss = 0.0

    for batch in tqdm(dataloader):
        # functional.reset_net(encoder)
        
        if use_mask:
            video_batch, mel_batch, lengths = batch
            lengths = lengths.to(device)
        else:
            video_batch, mel_batch = batch
            lengths = None
        
        if video_batch.shape[2] >= 125:
            print(f"  -> Bỏ qua batch vì T = {video_batch.shape[2]} >= 125")
            continue

        video_batch = video_batch.to(device)
        mel_batch = mel_batch.to(device)

        optimizer.zero_grad()

        z = encoder(video_batch) # 
        mel_pred = decoder(z)

        if use_mask and lengths is not None:
            T_max = mel_batch.shape[1]
            mask = torch.arange(T_max, device=device).unsqueeze(0) < lengths.unsqueeze(1)
            loss = criterion(mel_batch[mask], mel_pred[mask])
        else:
            loss = criterion(mel_batch, mel_pred)

        loss.backward()

        if max_grad_norm:
            torch.nn.utils.clip_grad_norm_(
                list(encoder.parameters()) + list(decoder.parameters()),
                max_grad_norm
            )

        optimizer.step()

        total_loss += loss.item()
        functional.reset_net(encoder)
        del loss, z, mel_pred, video_batch, mel_batch
    
    avg_loss = total_loss / len(dataloader)

    return avg_loss


In [25]:
optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=1e-4
)
criterion = nn.L1Loss()

avg_loss = train_one_epoch(
    encoder, decoder, dataloader, optimizer, criterion, device,
    use_mask=True, max_grad_norm=1.0
)
print(f"Loss", avg_loss)

  0%|          | 0/50 [00:00<?, ?it/s]

  -> Bỏ qua batch vì T = 269 >= 200
  -> Bỏ qua batch vì T = 207 >= 200
  -> Bỏ qua batch vì T = 210 >= 200
  -> Bỏ qua batch vì T = 228 >= 200
  -> Bỏ qua batch vì T = 209 >= 200
  -> Bỏ qua batch vì T = 227 >= 200
  -> Bỏ qua batch vì T = 334 >= 200
  -> Bỏ qua batch vì T = 363 >= 200
  -> Bỏ qua batch vì T = 278 >= 200
  -> Bỏ qua batch vì T = 214 >= 200
  -> Bỏ qua batch vì T = 278 >= 200
Loss 36.62863792419434


In [25]:
def train_full(encoder, decoder, train_loader, optimizer, criterion, device,
               num_epochs=50, use_mask=True, max_grad_norm=1.0, 
               checkpoint_dir="checkpoints", save_best=True):
    """
    Huấn luyện nhiều epoch, in log loss, lưu checkpoint.
    """
    os.makedirs(checkpoint_dir, exist_ok=True)
    best_loss = float('inf')
    history = []

    for epoch in range(1, num_epochs + 1):
        # Huấn luyện 1 epoch
        avg_loss = train_one_epoch(encoder, decoder, train_loader,
                                   optimizer, criterion, device,
                                   use_mask=use_mask,
                                   max_grad_norm=max_grad_norm)
        history.append(avg_loss)

        # In log loss
        print(f"Epoch {epoch:3d}/{num_epochs} | Train Loss: {avg_loss:.6f}")

        # Lưu checkpoint định kỳ
        if epoch % 10 == 0 or epoch == num_epochs:
            checkpoint_path = os.path.join(checkpoint_dir, f"epoch_{epoch}.pth")
            torch.save({
                'epoch': epoch,
                'encoder_state_dict': encoder.state_dict(),
                'decoder_state_dict': decoder.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss,
            }, checkpoint_path)
            print(f"  -> Đã lưu checkpoint: {checkpoint_path}")

        # Lưu model tốt nhất
        if save_best and avg_loss < best_loss:
            best_loss = avg_loss
            best_path = os.path.join(checkpoint_dir, "best_model.pth")
            torch.save({
                'epoch': epoch,
                'encoder_state_dict': encoder.state_dict(),
                'decoder_state_dict': decoder.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': best_loss,
            }, best_path)
            print(f"  -> Đã lưu model tốt nhất: {best_path}")

    print("Hoàn tất huấn luyện!")
    return history

In [ ]:
optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=1e-4
)
criterion = nn.L1Loss()

history = train_full(encoder, decoder, dataloader, optimizer, criterion, device,
                      num_epochs=100, use_mask=True, max_grad_norm=1.0,
                      checkpoint_dir="my_checkpoints", save_best=True)

## Evaluate test

In [24]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Encoder params: {count_parameters(encoder):,}")
print(f"Decoder params: {count_parameters(decoder):,}")

Encoder params: 26,564,527
Decoder params: 1,597,264


In [24]:
%pip install speechbrain


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
from speechbrain.inference.vocoders import HIFIGAN

In [32]:


# 1. Khởi tạo Vocoder (sẽ tự động tải model về nếu chưa có)
# Sử dụng model 16kHz để khớp với dữ liệu của bạn
hifi_gan = HIFIGAN.from_hparams(
    source="speechbrain/tts-hifigan-libritts-16kHz",
    savedir="pretrained_models/tts-hifigan-libritts-16kHz" 
) 

encoder.eval()
decoder.eval()
with torch.no_grad():
    v, m, l = next(iter(dataloader))
    v = v.to(device)
    functional.reset_net(encoder)
    mel_pred = decoder(encoder(v))
    # Lấy clip đầu tiên trong batch, bỏ padding
    # mel_pred_np = mel_pred[0, :l[0]].cpu().numpy().T  # về shape (80, T)
    # Dùng vocoder (ví dụ HiFi‑GAN pretrained) để chuyển mel sang waveform
    # Sau đó lưu file wav và nghe.
    mel_specs = mel_pred.permute(0, 2, 1) 
    mel_specs = mel_specs.to('cpu') # Chuyển về CPU

# 3. Chuyển đổi Mel-spectrogram thành Waveform
# decode_batch trả về tensor có shape (1, 1, SAMPLE_SIZE)
    waveforms = hifi_gan.decode_batch(mel_specs) 

# 4. Lưu file âm thanh
    import torchaudio
    waveform_to_save = waveforms.squeeze(1).cpu() # (1, SAMPLE_SIZE)
    torchaudio.save('reconstructed_audio.wav', waveform_to_save, 16000)
    print("Đã lưu file âm thanh thành công!")

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/pretrained_models/tts-hifigan-libritts-16kHz/hyperparams.yaml'
INFO:speechbrain.utils.fetching:Fetch generator.ckpt: Using symlink found at '/content/pretrained_models/tts-hifigan-libritts-16kHz/generator.ckpt'
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: generator


Đã lưu file âm thanh thành công!


In [29]:
import IPython.display as ipd
import torchaudio

# Đọc file lên và phát lại
audio, sr = torchaudio.load('reconstructed_audio.wav')
print(f"Tần số lấy mẫu: {sr} Hz")
print(f"Độ dài: {audio.shape[-1]/sr:.2f} giây")
ipd.Audio(audio.numpy(), rate=sr)

Tần số lấy mẫu: 16000 Hz
Độ dài: 2.14 giây


In [ ]:

hifi_gan = HIFIGAN.from_hparams(
    source="speechbrain/tts-hifigan-libritts-16kHz",
    savedir="pretrained_models/tts-hifigan-libritts-16kHz" 
) 
# Lấy một batch ground truth Mel từ dataloader

v, m, l = next(iter(dataloader))
# Cần đưa về dạng (batch, n_mels, time) cho HiFi-GAN
mel_true = m.permute(0, 2, 1).cpu()  # (B, n_mels, T)
waveforms = hifi_gan.decode_batch(mel_true)
torchaudio.save('true_audio.wav', waveforms.squeeze(1).cpu(), 16000)

In [35]:
import IPython.display as ipd
import torchaudio

# Đọc file lên và phát lại
audio, sr = torchaudio.load('true_audio.wav')
print(f"Tần số lấy mẫu: {sr} Hz")
print(f"Độ dài: {audio.shape[-1]/sr:.2f} giây")
ipd.Audio(audio.numpy(), rate=sr)

Tần số lấy mẫu: 16000 Hz
Độ dài: 2.61 giây


# New approach: TFiLM WIRE DECODER

## FiLM + WIRE

In [ ]:
class FiLMWIRE(nn.Module):
    def __init__(self, in_features, out_features, omega_zero=30.0, scale=1.0, is_first=False):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.omega_zero = omega_zero
        self.scale = scale
        self.is_first = is_first
        with torch.no_grad():
            if self.is_first:
                bound = 1 / in_features
            else:
                bound = torch.sqrt(torch.tensor(6.0 / in_features)) / self.omega_zero
            self.linear.weight.uniform_(-bound, bound)

    def forward(self, X, gamma, beta):
        X = self.linear(X)
        if not self.is_first:
            X = self.omega_zero * X 
        modulated = gamma * X + beta
        real = torch.cos(self.omega_zero * modulated) * torch.exp(-modulated**2 / (2 * self.scale**2))
        return real


## FiLM WIRE DECODER

In [ ]:
class TFiLMWIREDecoder(nn.Module):
    def __init__(self, condition_dim=512, hidden_dim=256, out_dim=80, num_layers=4, omega_zero=30.0, scale=5.0, use_conv=False):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        total_params = num_layers * 2 * hidden_dim
        if use_conv:
            self.param_net = nn.Conv1d(condition_dim, total_params, kernel_size=3, padding=1)
            nn.init.zeros_(self.param_net.weight)
            with torch.no_grad():
                for i in range(num_layers):
                    start = i * 2 * hidden_dim
                    self.param_net.bias[start:start + hidden_dim].fill_(0.0)
                    self.param_net.bias[start + hidden_dim:start + 2 * hidden_dim].fill_(1.0)
        else:
            self.param_net = nn.Sequential(
                nn.Linear(condition_dim, condition_dim),
                nn.ReLU(),
                nn.Linear(condition_dim, total_params)
            )
            nn.init.zeros_(self.param_net[-1].weight)
            with torch.no_grad():
                for i in range(num_layers):
                    start = i * 2  * hidden_dim
                    self.param_net[-1].bias[start:start + hidden_dim].fill_(0.0)
                    self.param_net[-1].bias[start + hidden_dim:start + 2 * hidden_dim].fill_(1.0)

        self.wire_layers = nn.ModuleList()
        self.wire_layers.append(FiLMWIRE(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, scale=scale, is_first=True))
        for _ in range(1, num_layers):
            self.wire_layers.append(FiLMWIRE(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, scale=scale, is_first=False))
        self.final_layer = nn.Linear(hidden_dim , out_dim)
        nn.init.xavier_uniform_(self.final_layer.weight)
        nn.init.zeros_(self.final_layer.bias)

        self.input_constant = nn.Parameter(torch.randn(1, hidden_dim) * 0.1)
    
    def forward(self, Condition):
        B, T, _ = Condition.shape
        if isinstance(self.param_net, nn.Conv1d):
            params = self.param_net(Condition.permute(0, 2, 1))
            params = params.permute(0, 2, 1)
        else:
            params = self.param_net(Condition.reshape(B * T, -1))
            params = params.reshape(B, T, -1)
        
        gammas = []
        betas = []

        for i in range(self.num_layers):
            start = i * 2 * self.hidden_dim
            beta = params[:, :, start: start + self.hidden_dim]
            gamma = params[:, :, start + self.hidden_dim: start + 2 * self.hidden_dim]
            gammas.append(gamma)
            betas.append(beta)
        
        X = self.input_constant.expand(B * T, -1) # B*T, hidden_dim
        gammas_flat = [g.reshape(B * T, -1) for g in gammas]
        betas_flat = [b.reshape(B * T, -1) for b in betas]

        for i, layer in enumerate(self.wire_layers):
            X = layer(X, gammas_flat[i], betas_flat[i])
        out = self.final_layer(X)
        out = out.reshape(B, T, -1)
        return out

# New approach: WIRE + SIREN DUAL DECODER

## Dual Layer

In [ ]:
class DualFiLMLayer(nn.Module):
    def __init__(self, in_features, out_features, omega_siren=30.0, omega_wire=30.0, scale=5.0, is_first=False):
        super().__init__()
        self.siren = FiLMSIREN(in_features, out_features, omega_siren, is_first)
        self.wire = FiLMWIRE(in_features, out_features, omega_wire, is_first)
        self.fusion = nn.Sequential(
            nn.Linear(out_features * 2, out_features),
            nn.ReLU(),
            nn.Linear(out_features, out_features)
        )

    def forward(self, X, gamma_s, beta_s, gamma_w, beta_w):
        out_siren = self.siren(X, gamma_s, beta_s)
        out_wire = self.wire(X, gamma_w, beta_w)
        out = torch.cat([out_siren, out_wire], dim=-1)
        out = self.fusion(out)
        return out

## Dual Decoder

In [ ]:
class DualDecoder(nn.Module):
    def __init__(self, condition_dim=512, hidden_dim=256, out_dim=80, num_layers=4, omega_siren=30.0, omega_wire=30.0, scale=5.0, use_conv=False):
        super().__init__()
        self.num_layers = num_frames
        self.hidden_dim = hidden_dim

        total_params = self.num_layers * 4 * hidden_dim
        if use_conv:
            self.param_net = nn.Conv1d(condition_dim, total_params, kernel_size=3, padding=1)
            nn.init.zeros_(self.param_net.weight)
            with torch.no_grad():
                for i in range(num_layers):
                    start = i * 4 * hidden_dim
                    self.param_net.bias[start: start + hidden_dim].fill_(0.0)
                    self.param_net.bias[start + hidden_dim: start + 2 * hidden_dim].fill_(1.0)
                    self.param_net.bias[start + 2 * hidden_dim: start + 3 * hidden_dim].fill_(0.0)
                    self.param_net.bias[start + 3 * hidden_dim: start + 4 * hidden_dim].fill_(1.0) 

        else: 
            self.param_net = nn.Sequential(
                nn.Linear(condition_dim, condition_dim),
                nn.ReLU(),
                nn.Linear(condition_dim, total_params)
            )
            nn.init.zeros_(self.param_net[-1].weight)
            with torch.no_grad():
                for i in range(num_layers):
                    start = i * 4 * hidden_dim
                    self.param_net[-1].bias[start: start + hidden_dim].fill_(0.0)
                    self.param_net[-1].bias[start + hidden_dim: start + 2 * hidden_dim].fill_(1.0)
                    self.param_net[-1].bias[start + 2 * hidden_dim: start + 3 * hidden_dim].fill_(0.0)
                    self.param_net[-1].bias[start + 3 * hidden_dim: start + 4 * hidden_dim].fill_(1.0)

        self.dual_layers = nn.ModuleList()
        self.dual_layers.append(DualFiLMLayer(hidden_dim, hidden_dim, omega_siren, omega_wire, scale, is_first=True))
        for _ in range(1, num_layers):
            self.dual_layers.append(DualFiLMLayer(hidden_dim, hidden_dim, omega_siren, omega_wire, scale, is_first=False))
        self.final_layer = nn.Linear(hidden_dim, out_dim)
        nn.init.xavier_uniform_(self.final_layer.weight)
        nn.init.zeros_(self.final_layer.bias)

        self.input_constant = nn.Parameter(torch.randn(1, hidden_dim) * 0.1)

    def forward(self, Condition):
        B, T, _ = Condition.shape
        if isinstance(self.param_net, nn.Conv1d):
            params = self.param_net(Condition.permute(0, 2, 1))
            params = params.permute(0, 2, 1)
        else:
            params = self.param_net(Condition.reshape(B * T, -1))
            params = params.reshape(B, T, -1)

        gammas_s = []
        betas_s = []
        gammas_w = []
        betas_w = []

        for i in range(self.num_layers):
            start = i * 4 * self.hidden_dim
            # SIREN
            beta_s = params[:, :, start: start + self.hidden_dim]
            gamma_s = params[:, :, start + self.hidden_dim: start + 2 * self.hidden_dim]

            # WIRE
            beta_w = params[:, :, start + 2 * self.hidden_dim: start + 3 * self.hidden_dim]
            gamma_w = params[:, :, start + 3 * self.hidden_dim: start + 4 * self.hidden_dim]

            gammas_s.append(gamma_s)
            betas_s.append(beta_s)
            gammas_w.append(gamma_w)
            betas_w.append(beta_w)

        X = self.input_constant.expand(B * T, -1)
        
        gammas_s_flat = [g.reshape(B * T, -1) for g in gammas_s]
        betas_s_flat = [b.reshape(B * T, -1) for b in betas_s]
        gammas_w_flat = [g.reshape(B * T, -1) for g in gammas_w]
        betas_w_flat = [b.reshape(B * T, -1) for b in betas_w]

        for i, layer in enumerate(self.dual_layers):
            X = layer(
                gammas_s_flat[i], betas_s_flat[i],
                gammas_w_flat[i], betas_w_flat[i]
            )
        out = self.final_layer(X)
        out = out.reshape(B, T, -1)
        return out
            


# New approach: TFiLM FINER DECODER

## TFiLM FINER Layer

In [ ]:
class FiLMFINER(nn.Module):
    def __init__(self, in_features, out_features, omega_zero=30.0, is_first=False):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.omega_zero = omega_zero
        self.is_first = is_first
        with torch.no_grad():
            if self.is_first:
                bound = 1 / in_features
            else:
                bound = torch.sqrt(torch.tensor(6.0 / in_features)) / self.omega_zero
            self.linear.weight.uniform_(-bound, bound)
        
    def forward(self, gamma, beta):
        X = self.linear(X)
        if not self.is_first:
            X = self.omega_zero * X
        modulated = gamma * X + beta
        return torch.sin(self.omega_zero * (torch.abs(modulated) + 1.0) * modulated)

## TFiLM FINER DECODER

In [ ]:
class TFiLMFINERDecoder(nn.Module):
    def __init__(self, condition_dim=512, hidden_dim=256, out_dim=80, num_layers=4, omega_zero=30.0, use_conv=False):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        total_params = num_layers * 2 * hidden_dim
        if use_conv:
            self.param_net = nn.Conv1d(condition_dim, total_params, kernel_size=3, padding=1)
            nn.init.zeros_(self.param_net.weight)
            with torch.no_grad():
                for i in range(num_layers):
                    start = i * 2 * hidden_dim
                    self.param_net.bias[start:start + hidden_dim].fill_(0.0)
                    self.param_net.bias[start + hidden_dim:start + 2 * hidden_dim].fill_(1.0)
        else:
            self.param_net = nn.Sequential(
                nn.Linear(condition_dim, condition_dim),
                nn.ReLU(),
                nn.Linear(condition_dim, total_params)
            )
            nn.init.zeros_(self.param_net[-1].weight)
            with torch.no_grad():
                for i in range(num_layers):
                    start = i * 2  * hidden_dim
                    self.param_net[-1].bias[start:start + hidden_dim].fill_(0.0)
                    self.param_net[-1].bias[start + hidden_dim:start + 2 * hidden_dim].fill_(1.0)

        self.finer_layers = nn.ModuleList()
        self.finer_layers.append(FiLMFINER(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, is_first=True))
        for _ in range(1, num_layers):
            self.finer_layers.append(FiLMFINER(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, is_first=False))
        self.final_layer = nn.Linear(hidden_dim , out_dim)
        nn.init.xavier_uniform_(self.final_layer.weight)
        nn.init.zeros_(self.final_layer.bias)

        self.input_constant = nn.Parameter(torch.randn(1, hidden_dim) * 0.1)
    
    def forward(self, Condition):
        B, T, _ = Condition.shape
        if isinstance(self.param_net, nn.Conv1d):
            params = self.param_net(Condition.permute(0, 2, 1))
            params = params.permute(0, 2, 1)
        else:
            params = self.param_net(Condition.reshape(B * T, -1))
            params = params.reshape(B, T, -1)
        
        gammas = []
        betas = []

        for i in range(self.num_layers):
            start = i * 2 * self.hidden_dim
            beta = params[:, :, start: start + self.hidden_dim]
            gamma = params[:, :, start + self.hidden_dim: start + 2 * self.hidden_dim]
            gammas.append(gamma)
            betas.append(beta)
        
        X = self.input_constant.expand(B * T, -1) # B*T, hidden_dim
        gammas_flat = [g.reshape(B * T, -1) for g in gammas]
        betas_flat = [b.reshape(B * T, -1) for b in betas]

        for i, layer in enumerate(self.finer_layers):
            X = layer(X, gammas_flat[i], betas_flat[i])
        out = self.final_layer(X)
        out = out.reshape(B, T, -1)
        return out

# New approach: Wrap FINER SIREN DECODER

## FINER SIREN WRAP

In [ ]:
# out = sin((∣b~∣+1)⋅(ω0(Wx+b)))
class FiLMWrapFINSIREN(nn.Module):
    def __init__(self, in_features, out_features, omega_zero=30.0, is_first=False):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=True)
        self.omega_zero = omega_zero
        self.is_first = is_first
        self.freq_bias = nn.Parameter(torch.empty(out_features))
        self.reset_parameters()
    
    def reset_parameter(self):
        with torch.no_grad():
            if self.is_first:
                bound = 1.0 / self.linear.in_features
            else:
                bound = torch.sqrt(torch.tensor(6.0 / self.linear.in_features)) / self.omega_zero
            self.linear.weight.uniform_(-bound, bound)
            nn.init.zeros_(self.linear.bias)
            nn.init.uniform_(self.freq_bias, -10.0, 10.0)
    
    def forward(self, X, gamma, beta):
        X = self.linear(X)
        if not self.is_first:
            X = self.omega_zero * X
        modulated = gamma * X + beta
        scaled = torch.abs(self.freq_bias) + 1.0
        out = torch.sin(scaled.unsqueeze(0) * modulated)
        return out

## TFiLM WRAP FIN SIN

In [ ]:
class TFiLMWrapFISINDecoder(nn.Module):
    def __init__(self, condition_dim=512, hidden_dim=256, out_dim=80, num_layers=4, omega_zero=30.0, use_conv=False):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        total_params = num_layers * 2 * hidden_dim
        if use_conv:
            self.param_net = nn.Conv1d(condition_dim, total_params, kernel_size=3, padding=1)
            nn.init.zeros_(self.param_net.weight)
            with torch.no_grad():
                for i in range(num_layers):
                    start = i * 2 * hidden_dim
                    self.param_net.bias[start:start + hidden_dim].fill_(0.0)
                    self.param_net.bias[start + hidden_dim:start + 2 * hidden_dim].fill_(1.0)
        else:
            self.param_net = nn.Sequential(
                nn.Linear(condition_dim, condition_dim),
                nn.ReLU(),
                nn.Linear(condition_dim, total_params)
            )
            nn.init.zeros_(self.param_net[-1].weight)
            with torch.no_grad():
                for i in range(num_layers):
                    start = i * 2  * hidden_dim
                    self.param_net[-1].bias[start:start + hidden_dim].fill_(0.0)
                    self.param_net[-1].bias[start + hidden_dim:start + 2 * hidden_dim].fill_(1.0)

        self.wrap_layers = nn.ModuleList()
        self.wrap_layers.append(FiLMWrapFINSIREN(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, is_first=True))
        for _ in range(1, num_layers):
            self.wrap_layers.append(FiLMWrapFINSIREN(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, is_first=False))
        self.final_layer = nn.Linear(hidden_dim , out_dim)
        nn.init.xavier_uniform_(self.final_layer.weight)
        nn.init.zeros_(self.final_layer.bias)

        self.input_constant = nn.Parameter(torch.randn(1, hidden_dim) * 0.1)
    
    def forward(self, Condition):
        B, T, _ = Condition.shape
        if isinstance(self.param_net, nn.Conv1d):
            params = self.param_net(Condition.permute(0, 2, 1))
            params = params.permute(0, 2, 1)
        else:
            params = self.param_net(Condition.reshape(B * T, -1))
            params = params.reshape(B, T, -1)
        
        gammas = []
        betas = []

        for i in range(self.num_layers):
            start = i * 2 * self.hidden_dim
            beta = params[:, :, start: start + self.hidden_dim]
            gamma = params[:, :, start + self.hidden_dim: start + 2 * self.hidden_dim]
            gammas.append(gamma)
            betas.append(beta)
        
        X = self.input_constant.expand(B * T, -1) # B*T, hidden_dim
        gammas_flat = [g.reshape(B * T, -1) for g in gammas]
        betas_flat = [b.reshape(B * T, -1) for b in betas]

        for i, layer in enumerate(self.wrap_layers):
            X = layer(X, gammas_flat[i], betas_flat[i])
        out = self.final_layer(X)
        out = out.reshape(B, T, -1)
        return out

# New approach: Wrap FINER WIRE DECODER

## FINER WIRE WRAP

In [ ]:
class FiLMWrapFINWIRE(nn.Module):
    def __init__(self, in_features, out_features, omega_zero=30.0, scale=5.0, is_first=False):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.omega_zero = omega_zero
        self.scale = scale
        self.is_first = is_first
        self.freq_bias = nn.Parameter(torch.empty(out_features))
        with torch.no_grad():
            if self.is_first:
                bound = 1 / self.linear.in_features
            else:
                bound = torch.sqrt(torch.tensor(6.0 / in_features)) / self.omega_zero
            self.linear.weight.uniform_(-bound, bound)
            nn.init.zeros_(self.linear.bias)
            nn.init.uniform_(self.freq_bias, -10, 10)

    def forward(self, X, gamma, beta):
        X = self.linear(X)
        if not self.is_first:
            X = self.omega_zero * X 
        modulated = gamma * X + beta
        scaled = torch.abs(self.freq_bias) + 1.0
        real = torch.cos(scaled.unsqueeze(0) * modulated) * torch.exp(-modulated**2 / (2 * self.scale**2))
        return real


## TFiLM WRAP FIN WI DEOCDER

In [ ]:
class TFiLMWrapFIWIDecoder(nn.Module):
    def __init__(self, condition_dim=512, hidden_dim=256, out_dim=80, num_layers=4, omega_zero=30.0, scale=5.0, use_conv=False):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        total_params = num_layers * 2 * hidden_dim
        if use_conv:
            self.param_net = nn.Conv1d(condition_dim, total_params, kernel_size=3, padding=1)
            nn.init.zeros_(self.param_net.weight)
            with torch.no_grad():
                for i in range(num_layers):
                    start = i * 2 * hidden_dim
                    self.param_net.bias[start:start + hidden_dim].fill_(0.0)
                    self.param_net.bias[start + hidden_dim:start + 2 * hidden_dim].fill_(1.0)
        else:
            self.param_net = nn.Sequential(
                nn.Linear(condition_dim, condition_dim),
                nn.ReLU(),
                nn.Linear(condition_dim, total_params)
            )
            nn.init.zeros_(self.param_net[-1].weight)
            with torch.no_grad():
                for i in range(num_layers):
                    start = i * 2  * hidden_dim
                    self.param_net[-1].bias[start:start + hidden_dim].fill_(0.0)
                    self.param_net[-1].bias[start + hidden_dim:start + 2 * hidden_dim].fill_(1.0)

        self.wrap_layers = nn.ModuleList()
        self.wrap_layers.append(FiLMWrapFINWIRE(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, scale=scale, is_first=True))
        for _ in range(1, num_layers):
            self.wrap_layers.append(FiLMWrapFINWIRE(in_features=hidden_dim, out_features=hidden_dim, omega_zero=omega_zero, scale=scale, is_first=False))
        self.final_layer = nn.Linear(hidden_dim , out_dim)
        nn.init.xavier_uniform_(self.final_layer.weight)
        nn.init.zeros_(self.final_layer.bias)

        self.input_constant = nn.Parameter(torch.randn(1, hidden_dim) * 0.1)
    
    def forward(self, Condition):
        B, T, _ = Condition.shape
        if isinstance(self.param_net, nn.Conv1d):
            params = self.param_net(Condition.permute(0, 2, 1))
            params = params.permute(0, 2, 1)
        else:
            params = self.param_net(Condition.reshape(B * T, -1))
            params = params.reshape(B, T, -1)
        
        gammas = []
        betas = []

        for i in range(self.num_layers):
            start = i * 2 * self.hidden_dim
            beta = params[:, :, start: start + self.hidden_dim]
            gamma = params[:, :, start + self.hidden_dim: start + 2 * self.hidden_dim]
            gammas.append(gamma)
            betas.append(beta)
        
        X = self.input_constant.expand(B * T, -1) # B*T, hidden_dim
        gammas_flat = [g.reshape(B * T, -1) for g in gammas]
        betas_flat = [b.reshape(B * T, -1) for b in betas]

        for i, layer in enumerate(self.wrap_layers):
            X = layer(X, gammas_flat[i], betas_flat[i])
        out = self.final_layer(X)
        out = out.reshape(B, T, -1)
        return out

# New approach: Dual Wrap FINER - SIREN WIRE

## Dual Wrap Layer

In [ ]:
class DualWrapLayer(nn.Module):
    def __init__(self, in_features, out_features, omega_fisin=30.0, omega_fiwi=30.0, scale=5.0, is_first=False):
        super().__init__()
        self.fisin = FiLMWrapFINSIREN(in_features, out_features, omega_fisin, is_first)
        self.fiwi = FiLMWrapFINWIRE(in_features, out_features, omega_fiwi, is_first)
        self.fusion = nn.Sequential(
            nn.Linear(in_features * 2, out_features),
            nn.ReLU(),
            nn.Linear(out_features, out_features)
        )
    
    def forward(self, X, gamma_fs, beta_fs, gamma_fw, beta_fw):
        out_fs = self.fisin(X, gamma_fs, beta_fs)
        out_fw = self.fiwi(X, gamma_fw, beta_fw)
        out = torch.cat([out_fs, out_fw], dim=-1)
        out = self.fusion(out)
        return out

## Dual Wrap Decoder

In [ ]:
class DualWrapDecoder(nn.Module):
    def __init__(self, condition_dim=512, hidden_dim=256, out_dim=80, num_layers=4, omega_siren=30.0, omega_wire=30.0, scale=5.0, use_conv=False):
        super().__init__()
        self.num_layers = num_frames
        self.hidden_dim = hidden_dim

        total_params = self.num_layers * 4 * hidden_dim
        if use_conv:
            self.param_net = nn.Conv1d(condition_dim, total_params, kernel_size=3, padding=1)
            nn.init.zeros_(self.param_net.weight)
            with torch.no_grad():
                for i in range(num_layers):
                    start = i * 4 * hidden_dim
                    self.param_net.bias[start: start + hidden_dim].fill_(0.0)
                    self.param_net.bias[start + hidden_dim: start + 2 * hidden_dim].fill_(1.0)
                    self.param_net.bias[start + 2 * hidden_dim: start + 3 * hidden_dim].fill_(0.0)
                    self.param_net.bias[start + 3 * hidden_dim: start + 4 * hidden_dim].fill_(1.0) 

        else: 
            self.param_net = nn.Sequential(
                nn.Linear(condition_dim, condition_dim),
                nn.ReLU(),
                nn.Linear(condition_dim, total_params)
            )
            nn.init.zeros_(self.param_net[-1].weight)
            with torch.no_grad():
                for i in range(num_layers):
                    start = i * 4 * hidden_dim
                    self.param_net[-1].bias[start: start + hidden_dim].fill_(0.0)
                    self.param_net[-1].bias[start + hidden_dim: start + 2 * hidden_dim].fill_(1.0)
                    self.param_net[-1].bias[start + 2 * hidden_dim: start + 3 * hidden_dim].fill_(0.0)
                    self.param_net[-1].bias[start + 3 * hidden_dim: start + 4 * hidden_dim].fill_(1.0)

        self.dual_layers = nn.ModuleList()
        self.dual_layers.append(DualWrapLayer(hidden_dim, hidden_dim, omega_siren, omega_wire, scale, is_first=True))
        for _ in range(1, num_layers):
            self.dual_layers.append(DualWrapLayer(hidden_dim, hidden_dim, omega_siren, omega_wire, scale, is_first=False))
        self.final_layer = nn.Linear(hidden_dim, out_dim)
        nn.init.xavier_uniform_(self.final_layer.weight)
        nn.init.zeros_(self.final_layer.bias)

        self.input_constant = nn.Parameter(torch.randn(1, hidden_dim) * 0.1)

    def forward(self, Condition):
        B, T, _ = Condition.shape
        if isinstance(self.param_net, nn.Conv1d):
            params = self.param_net(Condition.permute(0, 2, 1))
            params = params.permute(0, 2, 1)
        else:
            params = self.param_net(Condition.reshape(B * T, -1))
            params = params.reshape(B, T, -1)

        gammas_s = []
        betas_s = []
        gammas_w = []
        betas_w = []

        for i in range(self.num_layers):
            start = i * 4 * self.hidden_dim
            # SIREN
            beta_s = params[:, :, start: start + self.hidden_dim]
            gamma_s = params[:, :, start + self.hidden_dim: start + 2 * self.hidden_dim]

            # WIRE
            beta_w = params[:, :, start + 2 * self.hidden_dim: start + 3 * self.hidden_dim]
            gamma_w = params[:, :, start + 3 * self.hidden_dim: start + 4 * self.hidden_dim]

            gammas_s.append(gamma_s)
            betas_s.append(beta_s)
            gammas_w.append(gamma_w)
            betas_w.append(beta_w)

        X = self.input_constant.expand(B * T, -1)
        gammas_s_flat = [g.reshape(B * T, -1) for g in gammas_s]
        betas_s_flat = [b.reshape(B * T, -1) for b in betas_s]
        gammas_w_flat = [g.reshape(B * T, -1) for g in gammas_w]
        betas_w_flat = [b.reshape(B * T, -1) for b in betas_w]

        for i, layer in enumerate(self.dual_layers):
            X = layer(
                gammas_s_flat[i], betas_s_flat[i],
                gammas_w_flat[i], betas_w_flat[i]
            )
        out = self.final_layer(X)
        out = out.reshape(B, T, -1)
        return out
            
